# Matching questionnaire responses to cleaned sessions

For each `Conversation ID:` in the questionnaire, find the session directory in the cleaned log set that the response belongs to.

**Logic.**  Multiple respondents sometimes share the same `Conversation ID:` (the experimenter reused IDs across consecutive sessions). After the cleaning notebook has removed test runs, aborts and failed interactions, the surviving sessions are paired with questionnaire rows **in time order**: for each ID, the first (earliest) questionnaire response is paired with the first (earliest) surviving session of that ID, the second with the second, and so on.

**Inputs.**

- `docs/thesis/experiment/results/results.csv` — questionnaire responses.
- `voice-agent/src/experiment/results/experiments_cleaned_manualy/` — sessions left after automatic + manual cleaning.

**Outputs.**

- `docs/thesis/experiment/results/matched_sessions.csv` — one row per questionnaire row, with the matched session path, the matched condition (A or B), and the match status.
- Statistics printed at the bottom of the notebook (matched / unmatched, A vs B split).

The downstream analysis notebook should read `matched_sessions.csv` to join subjective answers with their session logs.

## Section 0 — Paths

In [8]:
import csv
import json
import re
import collections
import pathlib
from datetime import datetime
from zoneinfo import ZoneInfo

import pandas as pd

# Resolve project root.
PROJECT_ROOT = pathlib.Path.cwd()
while not (PROJECT_ROOT / "docker").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

CSV_PATH    = PROJECT_ROOT / "docs/thesis/experiment/results/results.csv"
LOG_ROOT    = PROJECT_ROOT / "voice-agent/src/experiment/results/experiments_cleaned_manualy"
OUT_CSV     = PROJECT_ROOT / "docs/thesis/experiment/results/matched_sessions.csv"

TZ = ZoneInfo("Europe/Prague")

print(f"CSV     : {CSV_PATH}   (exists={CSV_PATH.exists()})")
print(f"LOG_ROOT: {LOG_ROOT}   (exists={LOG_ROOT.exists()})")
print(f"OUT_CSV : {OUT_CSV}")

CSV     : /home/lucas/Projects/FEL/Pepper/docs/thesis/experiment/results/results.csv   (exists=True)
LOG_ROOT: /home/lucas/Projects/FEL/Pepper/voice-agent/src/experiment/results/experiments_cleaned_manualy   (exists=True)
OUT_CSV : /home/lucas/Projects/FEL/Pepper/docs/thesis/experiment/results/matched_sessions.csv


## Section 1 — Load questionnaire and group by ID

For each row keep the original CSV row index (so the mapping can be joined back), the conv_id verbatim, the numeric ID (`T63` → `63`), and the submission timestamp parsed as Europe/Prague local time. Group by numeric ID and sort each group ascending by submission time.

In [9]:
raw = pd.read_csv(CSV_PATH)
print(f"Questionnaire rows: {len(raw)}")

q = pd.DataFrame({
    "csv_row":      raw.index,
    "conv_id":      raw["Conversation ID:"].astype(str),
    "submitted_at": pd.to_datetime(raw["Časová značka"], format="%d.%m.%Y %H:%M:%S").dt.tz_localize(TZ),
})
q["id_num"] = q["conv_id"].str.extract(r"(\d+)").astype("Int64")
q = q.sort_values(["id_num", "submitted_at"]).reset_index(drop=True)
# Position of each response within its own ID group (0-based)
q["nth_for_id"] = q.groupby("id_num").cumcount()

display(q)

Questionnaire rows: 26


,csv_row,conv_id,submitted_at,id_num,nth_for_id
0,1,T01,2026-05-18 09:09:55+02:00,1,0
1,25,K01,2026-05-18 20:16:35+02:00,1,1
2,3,T07,2026-05-18 09:31:45+02:00,7,0
3,2,T08,2026-05-18 09:27:21+02:00,8,0
4,4,T14,2026-05-18 10:16:30+02:00,14,0
5,0,T30,2026-05-15 15:43:01+02:00,30,0
6,5,T30,2026-05-18 10:51:05+02:00,30,1
7,6,T33,2026-05-18 11:01:50+02:00,33,0
8,7,T33,2026-05-18 11:42:04+02:00,33,1
9,8,T34,2026-05-18 11:43:18+02:00,34,0


## Section 2 — Index cleaned sessions and group by student ID

Walk every `student<ID>_streaming<VARIANT>_<HHMMSS>` directory under `LOG_ROOT`, extract `(date, student_id, variant, start_dt)`, group by student ID, and sort each group ascending by start time.

In [10]:
rows = []
for d in sorted(LOG_ROOT.glob("*/student*_streaming*_*")):
    if not d.is_dir():
        continue
    m = re.match(r"student(\d+)_streaming([AB])_(\d{6})$", d.name)
    if not m:
        continue
    sid = int(m.group(1))
    variant = m.group(2)
    hhmmss = m.group(3)
    date_str = d.parent.name
    start_dt = datetime.strptime(f"{date_str} {hhmmss}", "%Y-%m-%d %H%M%S").replace(tzinfo=TZ)
    rows.append({
        "student_id":  sid,
        "variant":     variant,
        "start_dt":    start_dt,
        "dir_rel":     str(d.relative_to(LOG_ROOT)),
    })

sessions = pd.DataFrame(rows).sort_values(["student_id", "start_dt"]).reset_index(drop=True)
sessions["nth_for_id"] = sessions.groupby("student_id").cumcount()
print(f"Cleaned sessions: {len(sessions)}")
display(sessions)

Cleaned sessions: 20


,student_id,variant,start_dt,dir_rel,nth_for_id
0,1,A,2026-05-18 09:03:14+02:00,2026-05-18/student1_streamingA_090314,0
1,7,A,2026-05-18 09:18:42+02:00,2026-05-18/student7_streamingA_091842,0
2,8,B,2026-05-18 09:19:46+02:00,2026-05-18/student8_streamingB_091946,0
3,14,B,2026-05-18 10:05:23+02:00,2026-05-18/student14_streamingB_100523,0
4,30,B,2026-05-18 10:44:29+02:00,2026-05-18/student30_streamingB_104429,0
5,33,A,2026-05-18 11:31:46+02:00,2026-05-18/student33_streamingA_113146,0
6,34,B,2026-05-18 11:34:24+02:00,2026-05-18/student34_streamingB_113424,0
7,34,B,2026-05-18 11:48:16+02:00,2026-05-18/student34_streamingB_114816,1
8,38,B,2026-05-18 11:56:44+02:00,2026-05-18/student38_streamingB_115644,0
9,40,B,2026-05-18 12:06:10+02:00,2026-05-18/student40_streamingB_120610,0


## Section 3 — Ordinal pairing

For each questionnaire row, look up its `(id_num, nth_for_id)` in the sessions table. If a session exists at that position, it is the match; otherwise the row is left unmatched.

In [11]:
sess_idx = sessions.set_index(["student_id", "nth_for_id"])

records = []
for _, row in q.iterrows():
    key = (row["id_num"], row["nth_for_id"])
    if pd.isna(row["id_num"]):
        records.append({**row, "matched_dir": None, "matched_variant": None,
                        "matched_start": None, "match_status": "no_id_number"})
        continue
    key = (int(row["id_num"]), int(row["nth_for_id"]))
    if key in sess_idx.index:
        s = sess_idx.loc[key]
        records.append({
            "csv_row":         row["csv_row"],
            "conv_id":         row["conv_id"],
            "submitted_at":    row["submitted_at"],
            "id_num":          row["id_num"],
            "nth_for_id":      row["nth_for_id"],
            "matched_dir":     s["dir_rel"],
            "matched_variant": s["variant"],
            "matched_start":   s["start_dt"],
            "match_status":    "matched",
        })
    else:
        records.append({
            "csv_row":         row["csv_row"],
            "conv_id":         row["conv_id"],
            "submitted_at":    row["submitted_at"],
            "id_num":          row["id_num"],
            "nth_for_id":      row["nth_for_id"],
            "matched_dir":     None,
            "matched_variant": None,
            "matched_start":   None,
            "match_status":    "no_session",
        })

matches = pd.DataFrame(records)
# Restore original CSV order for readability
matches = matches.sort_values("csv_row").reset_index(drop=True)
print(f"Built {len(matches)} match records.")

Built 26 match records.


## Section 4 — Export the matching as CSV

In [12]:
matches.to_csv(OUT_CSV, index=False)
print(f"Wrote {OUT_CSV}  ({OUT_CSV.stat().st_size} bytes)")

Wrote /home/lucas/Projects/FEL/Pepper/docs/thesis/experiment/results/matched_sessions.csv  (2657 bytes)


## Section 5 — Display the matching table

Reload the CSV we just wrote (so the cell that downstream notebooks will execute also runs here, end-to-end) and display it.

In [13]:
loaded = pd.read_csv(OUT_CSV)
display(loaded)

,csv_row,conv_id,submitted_at,id_num,nth_for_id,matched_dir,matched_variant,matched_start,match_status
0,0,T30,2026-05-15 15:43:01+02:00,30,0,2026-05-18/student30_streamingB_104429,B,2026-05-18 10:44:29+02:00,matched
1,1,T01,2026-05-18 09:09:55+02:00,1,0,2026-05-18/student1_streamingA_090314,A,2026-05-18 09:03:14+02:00,matched
2,2,T08,2026-05-18 09:27:21+02:00,8,0,2026-05-18/student8_streamingB_091946,B,2026-05-18 09:19:46+02:00,matched
3,3,T07,2026-05-18 09:31:45+02:00,7,0,2026-05-18/student7_streamingA_091842,A,2026-05-18 09:18:42+02:00,matched
4,4,T14,2026-05-18 10:16:30+02:00,14,0,2026-05-18/student14_streamingB_100523,B,2026-05-18 10:05:23+02:00,matched
5,5,T30,2026-05-18 10:51:05+02:00,30,1,NaN,NaN,NaN,no_session
6,6,T33,2026-05-18 11:01:50+02:00,33,0,2026-05-18/student33_streamingA_113146,A,2026-05-18 11:31:46+02:00,matched
7,7,T33,2026-05-18 11:42:04+02:00,33,1,NaN,NaN,NaN,no_session
8,8,T34,2026-05-18 11:43:18+02:00,34,0,2026-05-18/student34_streamingB_113424,B,2026-05-18 11:34:24+02:00,matched
9,9,T38,2026-05-18 12:08:05+02:00,38,0,2026-05-18/student38_streamingB_115644,B,2026-05-18 11:56:44+02:00,matched


## Section 6 — Statistics

Match coverage overall, A vs B split among matched rows, and a list of any unmatched rows so they can be inspected by hand.

In [14]:
print(f"=== Match coverage ({len(matches)} questionnaire rows) ===")
status_counts = matches["match_status"].value_counts()
for k, v in status_counts.items():
    print(f"  {k:15s} {v:3d}")

print()
print("=== Condition split (matched only) ===")
matched_only = matches[matches["match_status"] == "matched"]
if len(matched_only) > 0:
    variant_counts = matched_only["matched_variant"].value_counts()
    for v in ["A", "B"]:
        n = int(variant_counts.get(v, 0))
        pct = 100.0 * n / len(matched_only)
        print(f"  Condition {v}: {n:3d}  ({pct:.0f}%)")
    print(f"  TOTAL    : {len(matched_only):3d}")
else:
    print("  (no matched rows)")

print()
print("=== Unmatched rows (inspect by hand) ===")
unmatched = matches[matches["match_status"] != "matched"]
if len(unmatched) == 0:
    print("  (none)")
else:
    display(unmatched[["csv_row", "conv_id", "submitted_at", "match_status"]])

print()
print("=== Sessions in the cleaned set that no questionnaire row claimed ===")
matched_dirs = set(matched_only["matched_dir"].dropna())
orphan = sessions[~sessions["dir_rel"].isin(matched_dirs)]
if len(orphan) == 0:
    print("  (every cleaned session was claimed by a questionnaire row)")
else:
    display(orphan[["student_id", "variant", "start_dt", "dir_rel"]])

=== Match coverage (26 questionnaire rows) ===
  matched          20
  no_session        6

=== Condition split (matched only) ===
  Condition A:  10  (50%)
  Condition B:  10  (50%)
  TOTAL    :  20

=== Unmatched rows (inspect by hand) ===


,csv_row,conv_id,submitted_at,match_status
5,5,T30,2026-05-18 10:51:05+02:00,no_session
7,7,T33,2026-05-18 11:42:04+02:00,no_session
15,15,T47,2026-05-18 15:09:10+02:00,no_session
16,16,T47,2026-05-18 15:25:43+02:00,no_session
24,24,T63,2026-05-18 17:35:44+02:00,no_session
25,25,K01,2026-05-18 20:16:35+02:00,no_session



=== Sessions in the cleaned set that no questionnaire row claimed ===
  (every cleaned session was claimed by a questionnaire row)
